# Template

In [10]:
import polars as pl
from black.trans import defaultdict

import src.social_groups.polars_columns as plc
from social_groups.analysis.defs.notebooks.definitions import register_materialization
from social_groups.analysis.polars_transformations.apply_parsing_and_group_decision import (
    apply_parsing_and_group_decision,
)
from social_groups.reporting.group_reply import (
    GroupReplyAggregator,
    MajorityVote,
)
from social_groups.reporting.parsing import (
    AnswerComparer,
    AnswerOptions,
    AnswerParser,
)

In [2]:
parser = AnswerParser(AnswerOptions.letters_A_to_J)
group_reply = GroupReplyAggregator(MajorityVote())
comparer = AnswerComparer(
    AnswerOptions.letters_A_to_J, triple_underscore_handling="wrong"
)

In [3]:
from social_groups.analysis.definitions import defs

frame = defs().load_asset_value("changed_order_mad")

/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/.venv/lib/python3.11/site-packages/dagster/_config/pythonic_config/typing_utils.py:111: UserWarning: Field name "extension" in "PolarsParquetIOManager" shadows an attribute in parent "BasePolarsUPathIOManager"
  return super().__new__(cls, name, bases, namespaces, **kwargs)
2026-03-02 12:05:27 +0800 - dagster - DEBUG - system - Loading file from: /Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/results/analysis/dagster/changed_order_mad.parquet using PolarsParquetIOManager...


In [8]:
data = (
    apply_parsing_and_group_decision(frame, parser, comparer, group_reply)
    .group_by("group_constellation")
    .agg(pl.col("is_correct").mean().alias(plc.accuracy))
    .sort("accuracy", descending=True)
)

register_materialization(
    "changed_order_mad_evaluation_table",
    data,
    "Accuracy per group constellation for different orderings of models using MAD.",
)

data

group_constellation,accuracy
str,f64
"""MHH""",0.75
"""HLH""",0.73
"""LHH""",0.72
"""HMH""",0.71
"""HHH""",0.71
…,…
"""LML""",0.47
"""MLL""",0.45
"""ML""",0.42


In [16]:
groups: dict[str, list[str]] = defaultdict(list)

for group in data["group_constellation"]:
    split = "".join(sorted(group))
    groups[split].append(group)

for group, parts in groups.items():
    group_data = data.filter(pl.col(plc.group_constellation).is_in(parts)).head()

    group_data.sort("accuracy")

    print(
        " -> ".join(
            f"{group_data['group_constellation'][i]} ({group_data['accuracy'][i]:.2f})"
            for i in range(len(group_data))
        )
    )

MHH (0.75) -> HMH (0.71) -> HHM (0.61)
HLH (0.73) -> LHH (0.72) -> HHL (0.64)
HHH (0.71)
MMH (0.71) -> MHM (0.68) -> HMM (0.67)
MLH (0.70) -> LMH (0.69) -> HLM (0.65) -> LHM (0.63) -> HML (0.62)
MH (0.69) -> HM (0.67)
HH (0.68)
LLH (0.67) -> LHL (0.52) -> HLL (0.49)
LH (0.64) -> HL (0.52)
LMM (0.60) -> MLM (0.58) -> MML (0.58)
MMM (0.60)
MM (0.58)
LLM (0.57) -> LML (0.47) -> MLL (0.45)
LM (0.54) -> ML (0.42)
LLL (0.32)
LL (0.31)


-> Very significant: "Smarter model last" -> "Better results

- MHH is even better than HHH